In [21]:
import os
import re
import fitz  # PyMuPDF
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [22]:
def legal_cleaning(text):
    """
    Generalized cleaning for Indian Law Reports (SCR/AIR).
    """
    # 1. Remove "Alphabet Ladders" (Standalone A-H on their own lines) [cite: 1, 6, 209]
    text = re.sub(r'^\s*[A-H]\s*$', '', text, flags=re.MULTILINE)

    # 2. Generalized SCR/Citation Headers 
    # Removes year-specific report markers like [2020] 1 S.C.R. or SUPREME COURT REPORTS
    patterns = [
        r'\[\d{4}\]\s*\d+\s*S\.C\.R\.', 
        r'SUPREME\s*COURT\s*REPORTS',
        r'CASE\s*LAW\s*REFERENCE',
        r'DIGEST\s*OF\s*CASES'
    ]
    for pattern in patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 3. Remove Standalone Page Numbers [cite: 108, 138, 189]
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Structural Cleanup
    text = re.sub(r'\n\s*\n', '\n\n', text) # Normalize double newlines
    text = re.sub(r' +', ' ', text) # Remove extra spaces
    text = "".join(char for char in text if char.isprintable() or char in ['\n', '\t'])
    # Final trim to ensure no trailing blank lines from the removed ladders
    text = "\n".join([line.strip() for line in text.split('\n') if line.strip()])

    return text.strip()

In [23]:
def convert_and_clean_worker(pdf_path, output_folder):
    """
    Worker function: Extracts -> Cleans -> Saves
    """
    try:
        # Maintain original filename but save as .txt
        base_name = os.path.splitext(os.path.basename(pdf_path))[0]
        output_path = os.path.join(output_folder, f"{base_name}.txt")
        
        # Check if already exists to allow for resuming interrupted jobs
        if os.path.exists(output_path):
            return True

        raw_text = ""
        with fitz.open(pdf_path) as doc:
            for page in doc:
                raw_text += page.get_text("text") + "\n"
        
        cleaned_text = legal_cleaning(raw_text)
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)
        return True
    except Exception:
        return False

In [24]:
def recursive_thread_process(root_input_folder, output_folder, workers=8):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder, exist_ok=True)

    pdf_tasks = []
    for root, dirs, files in os.walk(root_input_folder):
        if 'others' in dirs:
            dirs.remove('others') 
        if os.path.basename(root).lower() == 'english':
            for file in files:
                if file.lower().endswith(".pdf"):
                    pdf_tasks.append(os.path.normpath(os.path.join(root, file)))

    print(f"Discovered {len(pdf_tasks)} PDFs. Processing with ThreadPool...")

    # ThreadPool is much safer on Windows Legion laptops
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(convert_and_clean_worker, path, output_folder): path for path in pdf_tasks}
        
        for future in tqdm(as_completed(futures), total=len(pdf_tasks), desc="Processing"):
            future.result()

In [25]:
# --- Usage ---
if __name__ == "__main__":
    start_year = str(input("Enter start year: "))
    end_year = str(input("Enter end year: "))

    # This contains subfolders 2020, 2021, 2022...
    input_root = f"../../My_Datasets/SC_{start_year}-{end_year}" 
    output_dir = f"../../My_Datasets/Text_Datasets/texts_{start_year}-{end_year}"
    recursive_thread_process(input_root, output_dir)

Discovered 7952 PDFs. Processing with ThreadPool...


Processing: 100%|██████████| 7952/7952 [17:39<00:00,  7.50it/s]  
